# RLHF on Kaggle — run once and walk away

**Fire-and-forget.** Accelerator = **GPU**, Internet = **On**, then **Save & Run All (Commit)** (or headless via `scripts/run_on_kaggle.sh`). Works on **either** Kaggle GPU (T4 *or* P100).

Runs SFT → reward model → PPO → eval and writes **`RESULTS.md`** to the output.

In [ ]:
# Kaggle's base PyTorch dropped P100 (sm_60) support and API runs keep drawing a P100.
# torch 2.4.1 supports BOTH P100 (sm_60) and T4 (sm_75). Pin transformers to the same era as
# torch 2.4.1 (4.47): the latest 5.x/4.57 import 'Qwen2ForCausalLM' via torch symbols absent in 2.4.1.
!pip install -q 'transformers>=4.44,<4.48' 'datasets>=2.20' 'accelerate>=0.33' 'peft>=0.12' pytest
!pip install -q torch==2.4.1 --index-url https://download.pytorch.org/whl/cu121
import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if torch.cuda.is_available():
    (torch.randn(8, 8, device='cuda') @ torch.randn(8, 8, device='cuda')).sum().item()  # CUDA sanity
    # NOTE: is_bf16_supported() returns True on P100 via emulation — we force float32 below to avoid it.

In [ ]:
import os
REPO_URL = 'https://github.com/TheYellowDuck/RLHF-pipeline.git'
ATTACHED = '/kaggle/input/rlhf-pipeline'
if REPO_URL:
    !rm -rf /kaggle/working/rlhf-pipeline && git clone -q $REPO_URL /kaggle/working/rlhf-pipeline
elif os.path.exists(ATTACHED):
    !rm -rf /kaggle/working/rlhf-pipeline && cp -r $ATTACHED /kaggle/working/rlhf-pipeline
else:
    raise SystemExit('Set REPO_URL or attach the repo as a Kaggle Dataset.')
%cd /kaggle/working/rlhf-pipeline

In [ ]:
# Pre-flight: proves the pipeline is sound on this box (tiny model, ~1 min). Aborts the run if red.
!python scripts/smoke_test.py && python -m pytest tests/ -q

## Config — the only knobs

In [ ]:
MODEL   = 'Qwen/Qwen2.5-0.5B'
PRESET  = 'full'                # 'fast' (~2-3 h)  or  'full' (~6-8 h on fp32)
DATASET = 'uf'                  # 'uf' = UltraFeedback (cleaner, higher RM acc) | 'hh' = Anthropic/hh-rlhf

if DATASET == 'uf':
    DATA, TRAIN_SPLIT, EVAL_SPLIT = 'HuggingFaceH4/ultrafeedback_binarized', 'train_prefs', 'test_prefs'
else:
    DATA, TRAIN_SPLIT, EVAL_SPLIT = 'Anthropic/hh-rlhf', 'train', 'test'

if PRESET == 'fast':
    SFT_SAMPLES, RM_SAMPLES, PPO_EPISODES, MAX_NEW = 4000, 8000, 1024, 32
else:
    SFT_SAMPLES, RM_SAMPLES, PPO_EPISODES, MAX_NEW = 8000, 20000, 2048, 40
DTYPE = 'float32'               # P100 has no real bf16; fp32 is the safe path on either GPU
ROLLOUT, MINI, PPO_LR = 16, 2, '5e-7'   # fp32 on 16 GB: small minibatch keeps PPO in memory
print(f'model={MODEL}  preset={PRESET}  data={DATA}  dtype={DTYPE}  sft={SFT_SAMPLES}  rm={RM_SAMPLES}  ppo_ep={PPO_EPISODES}')

## 1. SFT  (fp32)

In [ ]:
!python scripts/train_sft.py \
  -o model.name_or_path=$MODEL -o model.dtype=$DTYPE -o data.name=$DATA -o data.train_split=$TRAIN_SPLIT -o data.eval_split=$EVAL_SPLIT \
  -o data.max_samples=$SFT_SAMPLES -o train.batch_size=8 -o train.bf16=false -o output_dir=checkpoints/sft

## 2. Reward model  (init from SFT + label-smoothed loss; batch 4 × grad-accum 2 = eff. 8)

In [ ]:
!python scripts/train_reward_model.py \
  -o model.name_or_path=checkpoints/sft -o model.dtype=$DTYPE -o data.name=$DATA -o data.train_split=$TRAIN_SPLIT -o data.eval_split=$EVAL_SPLIT \
  -o data.max_samples=$RM_SAMPLES -o data.max_eval_samples=500 \
  -o train.batch_size=4 -o train.grad_accum=2 -o train.bf16=false -o output_dir=checkpoints/reward_model

## 3. PPO (RL against the reward model; fp32, mini-batch 2 to fit 16 GB)

In [ ]:
!python scripts/train_ppo.py \
  -o policy.name_or_path=checkpoints/sft -o policy.dtype=$DTYPE \
  -o reward_model.name_or_path=checkpoints/reward_model -o reward_model.dtype=$DTYPE \
  -o data.name=$DATA -o data.train_split=$TRAIN_SPLIT \
  -o ppo.total_episodes=$PPO_EPISODES -o ppo.rollout_batch_size=$ROLLOUT -o ppo.mini_batch_size=$MINI \
  -o ppo.lr=$PPO_LR -o ppo.normalize_rewards=true \
  -o generation.max_new_tokens=$MAX_NEW -o data.max_prompt_length=160

## 4. Results → RESULTS.md

In [ ]:
import subprocess
def run(cmd):
    print('$', cmd, flush=True)
    o = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(o.stdout)
    if o.returncode: print(o.stderr[-1500:])
    return o.stdout

rm  = run(f'python scripts/evaluate.py rm-accuracy --reward-model checkpoints/reward_model --data {DATA} --split {EVAL_SPLIT} --max-samples 500')
win = run(f'python scripts/evaluate.py score-policy --policy checkpoints/ppo --reward-model checkpoints/reward_model --compare checkpoints/sft --data {DATA} --split {TRAIN_SPLIT} --num 100 --max-new-tokens {MAX_NEW}')
md = (f'# RLHF run results\n\n- model: `{MODEL}`\n- dataset: `{DATA}`\n- preset: `{PRESET}`\n\n'
      f'## Reward-model accuracy (held-out)\n```\n{rm}\n```\n\n'
      f'## PPO vs SFT — reward-model win-rate + sample completions\n```\n{win}\n```\n')
open('/kaggle/working/RESULTS.md','w').write(md)
print('\nSaved -> /kaggle/working/RESULTS.md')

### When it finishes
Output → **`RESULTS.md`** + `rlhf-pipeline/checkpoints/`. RM accuracy ~0.66–0.72 is realistic at this scale.
Still OOM on PPO? add `-o policy.use_lora=true`. PPO can resume after preemption with `--resume`.